# Phase 3 — Demo B: cross-regime transfer (2018 → 2019)


In [ ]:
# --- Setup: mount Drive + install driftbench from the public repo ---
from google.colab import drive  # noqa
drive.mount('/content/drive')

# Install the pinned commit for reproducibility (replace USERNAME and the ref).
!pip -q install "git+https://github.com/USERNAME/drift-conference.git@main"

import os
os.environ['DRIFT_DATA_ROOT'] = '/content/drive/MyDrive/drift-conference/data'
os.environ['DRIFT_REPO_ROOT'] = '/content/drift-conference'  # for results/manifests output
import driftbench as db; print('driftbench', db.__version__)


Train on 2018 (common-core features + family labels), test zero-shot on 2019, then recalibrate on a small time-ordered buffer (1/5/10%) of 2019 and re-test.

In [ ]:
import pandas as pd, numpy as np
from driftbench import (config, make_model, recalibrate, compute_metrics,
                        metrics_frame, time_ordered_buffer, shared_families)
from driftbench.splits import _time_order
A, B = 'CSE-CIC-IDS2018', 'CIC-DDoS2019'
Xa = pd.read_parquet(config.PROCESSED_DIR / A / 'features_core.parquet')
ya = pd.read_parquet(config.PROCESSED_DIR / A / 'labels_family.parquet')['family'].astype(str)
Xb = pd.read_parquet(config.PROCESSED_DIR / B / 'features_core.parquet')
yb = pd.read_parquet(config.PROCESSED_DIR / B / 'labels_family.parquet')['family'].astype(str)
metab = pd.read_parquet(config.INTERIM_DIR / B / 'metadata.parquet')
fams = shared_families(ya, yb)
ma = ya.isin(fams).values; mb = yb.isin(fams).values
Xa, ya = Xa[ma].reset_index(drop=True), ya[ma].reset_index(drop=True)
Xb, yb, metab = Xb[mb].reset_index(drop=True), yb[mb].reset_index(drop=True), metab[mb].reset_index(drop=True)
print('shared families:', fams)


In [ ]:
MODEL='rf'
base = make_model(MODEL, config.SEED_ANCHOR).fit(Xa, ya)
rows = {}
# zero-shot transfer
rows['zero_shot'] = compute_metrics(yb, base.predict(Xb), base.predict_proba(Xb), base.classes_)
# time-ordered calibration buffer sweep (recalibration only)
order = _time_order(metab, None)
for frac in (0.01, 0.05, 0.10):
    buf, rest = time_ordered_buffer(order, frac)
    cal = recalibrate(base, Xb.iloc[buf], yb.iloc[buf], method='sigmoid')
    rows[f'buffer_{int(frac*100)}pct'] = compute_metrics(
        yb.iloc[rest], cal.predict(Xb.iloc[rest]), cal.predict_proba(Xb.iloc[rest]), cal.classes_)
tbl = metrics_frame(rows); tbl.to_csv(config.RESULTS_DIR / f'demoB_transfer_{MODEL}.csv')
tbl[['macro_f1','mcc','ece']]
